# İstasyon Yükleyici — `deutschland_stationen.db`

DB REST API (`v6.db.transport.rest`) bazen 503 verdiği için istasyon verileri
otomatik önbelleğe alınamıyor. Bu notebook, tüm şehirleri toplu olarak sorgulayıp
`data/deutschland_stationen.db` dosyasına yazar.

Her şehir için **birden fazla arama anahtarı** kaydedilir (örn. hem `milan` hem
`milano`), böylece uygulama hangi isimle sorgularsa sorgulasın istasyonu bulur.

**Çalıştırma sırası:** Tüm hücreleri sırayla çalıştır.

In [1]:
import sqlite3
import time
import unicodedata
from pathlib import Path

import requests

DB_REST    = "https://v6.db.transport.rest"
STATION_DB = Path("..") / "data" / "deutschland_stationen.db"

def _norm(s: str) -> str:
    return (
        unicodedata.normalize("NFKD", s or "")
        .encode("ascii", "ignore")
        .decode()
        .lower()
        .strip()
    )

def _init_db(conn):
    conn.execute("""
        CREATE TABLE IF NOT EXISTS stationen (
            query      TEXT PRIMARY KEY,
            station_id TEXT NOT NULL,
            name       TEXT NOT NULL,
            lat        REAL,
            lon        REAL
        )
    """)
    for col in ("lat", "lon"):
        try:
            conn.execute(f"ALTER TABLE stationen ADD COLUMN {col} REAL")
        except sqlite3.OperationalError:
            pass
    conn.commit()

print("✅ Hazır")

✅ Hazır


In [2]:
# Şehir listesi: (db_anahtarları, api_sorgusu)
# db_anahtarları: DB'ye yazılacak tüm normalize isimler (uygulama hangisini kullanırsa kullansın)
# api_sorgusu  : DB REST'e gönderilecek gerçek sorgu
CITIES = [
    # ── Almanya ───────────────────────────────────────────────────────────
    (["berlin"],                       "Berlin"),
    (["munich", "munchen"],             "Munich"),
    (["hamburg"],                       "Hamburg"),
    (["frankfurt", "frankfurt am main"],"Frankfurt am Main"),
    (["cologne", "koln"],               "Cologne"),
    (["stuttgart"],                     "Stuttgart"),
    (["nuremberg", "nurnberg"],         "Nuremberg"),
    (["dusseldorf"],                    "Düsseldorf"),
    (["leipzig"],                       "Leipzig"),
    (["dresden"],                       "Dresden"),
    (["hannover"],                      "Hannover"),
    (["bremen"],                        "Bremen"),
    (["augsburg"],                      "Augsburg"),
    (["dortmund"],                      "Dortmund"),
    (["essen"],                         "Essen"),
    (["duisburg"],                      "Duisburg"),
    (["bochum"],                        "Bochum"),
    (["wuppertal"],                     "Wuppertal"),
    (["bielefeld"],                     "Bielefeld"),
    (["bonn"],                          "Bonn"),
    (["munster"],                       "Münster"),
    (["karlsruhe"],                     "Karlsruhe"),
    (["mannheim"],                      "Mannheim"),
    (["wiesbaden"],                     "Wiesbaden"),
    (["mainz"],                         "Mainz"),
    (["kiel"],                          "Kiel"),
    (["rostock"],                       "Rostock"),
    (["lubeck"],                        "Lübeck"),
    (["erfurt"],                        "Erfurt"),
    (["magdeburg"],                     "Magdeburg"),
    (["freiburg"],                      "Freiburg"),
    (["heidelberg"],                    "Heidelberg"),
    (["saarbrucken"],                   "Saarbrücken"),
    (["potsdam"],                       "Potsdam"),
    (["aachen"],                        "Aachen"),
    (["kassel"],                        "Kassel"),
    (["chemnitz"],                      "Chemnitz"),
    (["halle"],                         "Halle"),
    (["oberhausen"],                    "Oberhausen"),
    (["gelsenkirchen"],                 "Gelsenkirchen"),
    (["wolfsburg"],                     "Wolfsburg"),
    (["regensburg"],                    "Regensburg"),
    (["ingolstadt"],                    "Ingolstadt"),
    (["wurzburg"],                      "Würzburg"),
    (["ulm"],                           "Ulm"),
    (["braunschweig"],                  "Braunschweig"),
    (["osnabruck"],                     "Osnabrück"),
    (["darmstadt"],                     "Darmstadt"),
    (["paderborn"],                     "Paderborn"),
    (["bamberg"],                       "Bamberg"),
    (["bayreuth"],                      "Bayreuth"),
    (["passau"],                        "Passau"),
    (["landshut"],                      "Landshut"),
    (["rosenheim"],                     "Rosenheim"),
    (["konstanz", "constance"],         "Konstanz"),
    (["flensburg"],                     "Flensburg"),
    (["trier"],                         "Trier"),
    (["koblenz"],                       "Koblenz"),
    (["jena"],                          "Jena"),
    (["siegen"],                        "Siegen"),
    (["gottingen"],                     "Göttingen"),
    (["hildesheim"],                    "Hildesheim"),
    (["oldenburg"],                     "Oldenburg"),
    (["hagen"],                         "Hagen"),
    (["hamm"],                          "Hamm"),
    # ── Avusturya ─────────────────────────────────────────────────────────
    (["vienna", "wien"],                "Wien"),
    (["salzburg"],                      "Salzburg"),
    (["innsbruck"],                     "Innsbruck"),
    (["graz"],                          "Graz"),
    (["linz"],                          "Linz"),
    # ── İsviçre ───────────────────────────────────────────────────────────
    (["zurich", "zuerich"],             "Zürich"),
    (["basel"],                         "Basel"),
    (["bern"],                          "Bern"),
    (["geneva", "genf"],                "Genève"),
    (["lausanne"],                      "Lausanne"),
    (["lugano"],                        "Lugano"),
    (["lucerne", "luzern"],             "Luzern"),
    # ── Hollanda ──────────────────────────────────────────────────────────
    (["amsterdam"],                     "Amsterdam"),
    (["rotterdam"],                     "Rotterdam"),
    (["utrecht"],                       "Utrecht"),
    # ── Belçika ───────────────────────────────────────────────────────────
    (["brussels", "brussel", "bruxelles"], "Bruxelles"),
    (["antwerp", "antwerpen"],          "Antwerpen"),
    # ── Çek Cumhuriyeti ───────────────────────────────────────────────────
    (["prague", "prag", "praha"],       "Praha"),
    # ── Fransa ────────────────────────────────────────────────────────────
    (["paris"],                         "Paris"),
    (["strasbourg"],                    "Strasbourg"),
    (["lyon"],                          "Lyon"),
    # ── Danimarka ─────────────────────────────────────────────────────────
    (["copenhagen", "kobenhavn"],       "Copenhagen"),
    # ── Macaristan ────────────────────────────────────────────────────────
    (["budapest"],                      "Budapest"),
    # ── Polonya ───────────────────────────────────────────────────────────
    (["warsaw", "warszawa"],            "Warszawa"),
    (["krakow"],                        "Kraków"),
    (["wroclaw"],                       "Wrocław"),
    # ── İtalya (İtalyanca isimler — DB REST bu şekilde tanır) ─────────────
    (["milan", "milano"],              "Milano"),
    (["rome", "roma"],                 "Roma"),
    (["naples", "napoli"],             "Napoli"),
    (["venice", "venezia"],            "Venezia"),
    (["turin", "torino"],              "Torino"),
    (["florence", "firenze"],          "Firenze"),
    (["genoa", "genova"],              "Genova"),
    (["bologna"],                      "Bologna"),
    (["verona"],                       "Verona"),
    (["trieste"],                      "Trieste Centrale"),
    (["padua", "padova"],              "Padova"),
    (["bari"],                         "Bari"),
    (["palermo"],                      "Palermo"),
    (["catania"],                      "Catania"),
    (["messina"],                      "Messina"),
    (["parma"],                        "Parma"),
    (["modena"],                       "Modena"),
    (["brescia"],                      "Brescia"),
    (["bergamo"],                      "Bergamo"),
    (["treviso"],                      "Treviso"),
    (["vicenza"],                      "Vicenza"),
    (["trento"],                       "Trento"),
    (["bolzano"],                      "Bolzano"),
    (["udine"],                        "Udine"),
    (["ravenna"],                      "Ravenna"),
    (["pisa"],                         "Pisa"),
    (["ancona"],                       "Ancona"),
    (["perugia"],                      "Perugia"),
    (["lecce"],                        "Lecce"),
    (["taranto"],                      "Taranto"),
    (["brindisi"],                     "Brindisi"),
    (["pescara"],                      "Pescara"),
    (["salerno"],                      "Salerno"),
    (["cagliari"],                     "Cagliari"),
    (["foggia"],                       "Foggia"),
    (["reggio emilia"],                "Reggio Emilia"),
    (["reggio calabria"],              "Reggio Calabria"),
]

# Tüm anahtarları say
total_keys = sum(len(keys) for keys, _ in CITIES)
print(f"Şehir sayısı : {len(CITIES)}")
print(f"Toplam anahtar: {total_keys} (alias'lar dahil)")

Şehir sayısı : 128
Toplam anahtar: 152 (alias'lar dahil)


In [3]:
def lookup_station(query: str, session: requests.Session, retries: int = 3) -> dict | None:
    """DB REST üzerinden istasyon ara. Başarısız olursa None döner."""
    for attempt in range(retries):
        try:
            r = session.get(
                f"{DB_REST}/locations",
                params={"query": query, "results": 5, "stops": "true"},
                timeout=15,
                headers={"Accept": "application/json"},
            )
            if r.status_code == 503:
                wait = 2 ** attempt
                print(f"  ⚠️  503 — {wait}s bekleniyor (deneme {attempt+1}/{retries})")
                time.sleep(wait)
                continue
            r.raise_for_status()
            stops = [s for s in r.json() if s.get("type") in ("stop", "station")]
            if not stops:
                return None
            best = stops[0]
            loc  = best.get("location") or {}
            return {
                "station_id": str(best["id"]),
                "name":       best.get("name", query),
                "lat":        loc.get("latitude"),
                "lon":        loc.get("longitude"),
            }
        except Exception as e:
            print(f"  ❌ {query}: {e}")
            time.sleep(2 ** attempt)
    return None

print("✅ lookup_station hazır")

✅ lookup_station hazır


In [4]:
# Mevcut DB durumu — hangi anahtarlar zaten var?
with sqlite3.connect(STATION_DB) as conn:
    _init_db(conn)
    existing_keys = {row[0] for row in conn.execute("SELECT query FROM stationen")}

to_fetch = [
    (keys, api_query)
    for keys, api_query in CITIES
    if any(k not in existing_keys for k in keys)  # en az bir anahtar eksikse
]

print(f"DB'de mevcut anahtar : {len(existing_keys)}")
print(f"API'ye gidecek şehir : {len(to_fetch)}")
print(f"Atlanacak şehir      : {len(CITIES) - len(to_fetch)} (tüm anahtarları var)")

DB'de mevcut anahtar : 152
API'ye gidecek şehir : 0
Atlanacak şehir      : 128 (tüm anahtarları var)


In [5]:
from IPython.display import clear_output

ok, skip_count, fail = 0, len(CITIES) - len(to_fetch), 0
rows_to_write = []

with requests.Session() as session:
    for i, (keys, api_query) in enumerate(to_fetch, 1):
        clear_output(wait=True)
        print(f"İşleniyor {i}/{len(to_fetch)}: {api_query}  (anahtarlar: {keys})")
        print(f"  ✅ tamam={ok}  ⏭  atlandı={skip_count}  ❌ başarısız={fail}")

        data = lookup_station(api_query, session)
        if data:
            # Her anahtar için ayrı satır — hepsi aynı istasyona işaret eder
            for key in keys:
                rows_to_write.append((
                    key,
                    data["station_id"],
                    data["name"],
                    data["lat"],
                    data["lon"],
                ))
            ok += 1
            print(f"  → {data['name']}  id={data['station_id']}  "
                  f"lat={data['lat']}  lon={data['lon']}")
        else:
            fail += 1
            print(f"  ❌ Bulunamadı")

        # Her 10 şehirde bir kaydet
        if rows_to_write and i % 10 == 0:
            with sqlite3.connect(STATION_DB) as conn:
                conn.executemany(
                    "INSERT OR REPLACE INTO stationen "
                    "(query, station_id, name, lat, lon) VALUES (?,?,?,?,?)",
                    rows_to_write,
                )
                conn.commit()
            rows_to_write.clear()
            print(f"  💾 DB'ye yazıldı")

        time.sleep(0.3)

# Kalanları kaydet
if rows_to_write:
    with sqlite3.connect(STATION_DB) as conn:
        conn.executemany(
            "INSERT OR REPLACE INTO stationen "
            "(query, station_id, name, lat, lon) VALUES (?,?,?,?,?)",
            rows_to_write,
        )
        conn.commit()

clear_output(wait=True)
print("🏁 Tamamlandı!")
print(f"  ✅ Yüklendi : {ok} şehir")
print(f"  ⏭  Atlandı : {skip_count} şehir (zaten vardı)")
print(f"  ❌ Başarısız: {fail} şehir")

🏁 Tamamlandı!
  ✅ Yüklendi : 0 şehir
  ⏭  Atlandı : 128 şehir (zaten vardı)
  ❌ Başarısız: 0 şehir


In [6]:
# Son durum
import pandas as pd

with sqlite3.connect(STATION_DB) as conn:
    df = pd.read_sql("SELECT * FROM stationen ORDER BY name", conn)

print(f"DB toplam     : {len(df)} satır (alias'lar dahil)")
print(f"Benzersiz istasyon: {df['station_id'].nunique()}")
print(f"Koordinat var : {df['lat'].notna().sum()}  |  yok: {df['lat'].isna().sum()}")
df

DB toplam     : 152 satır (alias'lar dahil)
Benzersiz istasyon: 128
Koordinat var : 152  |  yok: 0


,query,station_id,name,lat,lon
0,aachen,8000001,Aachen Hbf,50.767640,6.091190
1,amsterdam,8400058,Amsterdam Centraal,52.379192,4.899427
2,ancona,8300186,Ancona,43.607597,13.497798
3,antwerp,8800007,Antwerpen Centraal,51.215810,4.421165
4,antwerpen,8800007,Antwerpen Centraal,51.215810,4.421165
...,...,...,...,...,...
147,wroclaw,5100069,Wroclaw Glowny,51.098076,17.037085
148,wuppertal,8000266,Wuppertal Hbf,51.254444,7.150155
149,wurzburg,8000260,Würzburg Hbf,49.802162,9.935930
150,zurich,8596008,ZURICH,47.378967,8.540534


## Bölüm 2 — Tren Güzergahı Arama Testi

Bu bölüm 3 soruyu yanıtlar:

1. **DB'deki istasyon ID'leri ile tren araması yapılabiliyor mu?**
2. **Sonuçlar otobüs/uçuş gibi önbelleğe (train_cache) yazılıp okunabiliyor mu?**
3. **Gerçek uygulama pipeline'ı (`get_trips`) uçtan uca çalışıyor mu?**

**Ön koşul:** Notebook'u proje kökünden (`multimodal-route-optimizer/`) aç
ya da aşağıdaki kurulum hücresi `sys.path`'i otomatik ayarlar.

In [7]:
# ─── Kurulum: backend modüllerini path'e ekle ─────────────────────────────
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import sqlite3
import unicodedata
from datetime import datetime, timedelta

import pandas as pd

STATION_DB = ROOT / "data" / "deutschland_stationen.db"
CACHE_DB   = ROOT / "data" / "search_cache.db"


def _norm(s: str) -> str:
    return (
        unicodedata.normalize("NFKD", s or "")
        .encode("ascii", "ignore")
        .decode()
        .lower()
        .strip()
    )


print("✅ Kurulum tamam — ROOT:", ROOT)

✅ Kurulum tamam — ROOT: C:\Users\esere\OneDrive\Belgeler\GitHub\multimodal-route-optimizer


In [8]:
# ─── Test 1: DB'deki istasyon ID'lerini doğrula ─────────────────────────────
# Hem Almanya hem İtalya hem de diğer Avrupa şehirleri
TEST_CITIES = [
    "Berlin", "Munich", "Hamburg", "Frankfurt", "Cologne",
    "Milan",  "Rome",   "Venice",  "Florence",  "Bologna",
    "Vienna", "Zurich", "Brussels", "Amsterdam",
]

header = f"{'Şehir':<20} {'Station ID':<12} {'Tam Ad':<35} {'Lat':<10} {'Lon':<10}"
print(header)
print("-" * len(header))

found, missing = [], []
with sqlite3.connect(STATION_DB) as conn:
    for city in TEST_CITIES:
        row = conn.execute(
            "SELECT station_id, name, lat, lon FROM stationen WHERE query=?",
            (_norm(city),),
        ).fetchone()
        if row:
            sid, name, lat, lon = row
            print(f"OK  {city:<20} {sid:<12} {name:<35} {str(lat):<10} {str(lon):<10}")
            found.append(city)
        else:
            print(f"MISSING  {city:<20}  -> DB'de YOK")
            missing.append(city)

print(f"\nSonuç: {len(found)}/{len(TEST_CITIES)} şehir bulundu", end="")
if missing:
    print(f"  |  Eksik: {missing}")
else:
    print(" — hepsi tamam!")

Şehir                Station ID   Tam Ad                              Lat        Lon       
-------------------------------------------------------------------------------------------
OK  Berlin               8011160      Berlin Hbf                          52.524925  13.369629 
OK  Munich               8096013      MUNICH (MÜNCHEN)                    48.134914  11.575383 
OK  Hamburg              8096009      HAMBURG                             53.55711   9.997434  
OK  Frankfurt            8096021      FRANKFURT(MAIN)                     50.107147  8.663785  
OK  Cologne              8096022      COLOGNE                             50.94131   6.967206  
OK  Milan                8396005      MILANO                              45.46248   9.207597  
OK  Rome                 8396004      ROMA                                41.901867  12.470654 
OK  Venice               8396008      VENEZIA                             45.444427  12.315491 
OK  Florence             8396001      FIRENZE   

In [9]:
# ─── Test 2: Station ID ile doğrudan tren güzergahı ara ───────────────────
# _get_trips_de() fonksiyonu DB'den station_id'yi çekip HAFAS API'ye gönderir
from backend.train_finder import _get_trips_de, _resolve_station_de

TOMORROW = (datetime.today() + timedelta(days=1)).strftime("%Y-%m-%d")

ORIGIN = "Berlin"
DEST   = "Rome"

print(f"Aranıyor: {ORIGIN} -> {DEST}  |  Tarih: {TOMORROW}")
print("=" * 60)

from_st = _resolve_station_de(ORIGIN)
to_st   = _resolve_station_de(DEST)
if from_st:
    print(f"Kaynak  : {from_st[1]}  (ID={from_st[0]})")
if to_st:
    print(f"Hedef   : {to_st[1]}  (ID={to_st[0]})")
print()

df_trips = _get_trips_de(ORIGIN, DEST, TOMORROW)

if not df_trips.empty:
    print(f"\n{len(df_trips)} tren secenegi bulundu:")
    display(df_trips[["departure_dt", "arrival_dt", "duration_min",
                       "price_eur", "stops", "provider"]].head(10))
else:
    print("Sonuc bulunamadi — API gecici olarak erisilelemez olabilir")

Aranıyor: Berlin -> Rome  |  Tarih: 2026-05-07
Kaynak  : Berlin Hbf  (ID=8011160)
Hedef   : ROMA  (ID=8396004)

✅ DE train search: 7 results Berlin Hbf→ROMA

7 tren secenegi bulundu:


,departure_dt,arrival_dt,duration_min,price_eur,stops,provider
0,2026-05-07 10:36:00,2026-05-08 06:35:00,1199,198.99,2,DB
1,2026-05-07 08:36:00,2026-05-07 23:30:00,894,206.99,2,DB
2,2026-05-07 08:31:00,2026-05-07 23:25:00,894,240.79,4,DB
3,2026-05-07 09:27:00,2026-05-08 06:21:00,1254,NaN,6,DB
4,2026-05-07 09:36:00,2026-05-08 06:21:00,1245,NaN,7,DB
5,2026-05-07 10:29:00,2026-05-08 06:23:00,1194,NaN,4,DB
6,2026-05-07 10:36:00,2026-05-08 06:23:00,1187,NaN,4,DB


In [10]:
# ─── Test 3: Önbellek (train_cache) yazma ve okuma testi ──────────────────
from backend import data_cache

print("=== ONBELLEK YAZMA ===")
if not df_trips.empty:
    data_cache.train_set(ORIGIN, DEST, TOMORROW, df_trips)
    print(f"train_cache'e yazildi  ({len(df_trips)} satir)")
else:
    print("UYARI: df_trips bos — once Test 2 hucresini calistir")

print()
print("=== ONBELLEK OKUMA ===")
cached = data_cache.train_get(ORIGIN, DEST, TOMORROW)
if cached is not None and not cached.empty:
    print(f"train_cache HIT — {len(cached)} satir geri alindi")
    display(cached[["departure_dt", "arrival_dt", "duration_min", "price_eur"]].head(5))
else:
    print("HATA: Onbellekten veri gelmedi")

=== ONBELLEK YAZMA ===
💾 train cache SAVE berlin→rome 2026-05-07 (7 rows)
train_cache'e yazildi  (7 satir)

=== ONBELLEK OKUMA ===
💾 train cache HIT  berlin→rome 2026-05-07
train_cache HIT — 7 satir geri alindi


,departure_dt,arrival_dt,duration_min,price_eur
0,2026-05-07 10:36:00,2026-05-08 06:35:00,1199,198.99
1,2026-05-07 08:36:00,2026-05-07 23:30:00,894,206.99
2,2026-05-07 08:31:00,2026-05-07 23:25:00,894,240.79
3,2026-05-07 09:27:00,2026-05-08 06:21:00,1254,None
4,2026-05-07 09:36:00,2026-05-08 06:21:00,1245,None


In [11]:
# ─── Test 4: Tam pipeline testi — get_trips() uçtan uca ───────────────────
# 1. çağrı API'ye gider; 2. çağrı mutlaka önbellekten gelmeli (cache HIT)
from backend.train_finder import get_trips

PAIR = ("Frankfurt", "Berlin")
DATE = TOMORROW

print(f"Arama: {PAIR[0]} -> {PAIR[1]}  |  Tarih: {DATE}")
print()
print("--- 1. CAGRI -----------------------------------------------")
df1 = get_trips(PAIR[0], PAIR[1], DATE)
n1 = len(df1) if df1 is not None and not df1.empty else 0
print(f"  Sonuc: {n1} sonuc")

print()
print("--- 2. CAGRI (onbellekten gelmeli) -------------------------")
df2 = get_trips(PAIR[0], PAIR[1], DATE)
n2 = len(df2) if df2 is not None and not df2.empty else 0
print(f"  Sonuc: {n2} sonuc")

print()
if df2 is not None and not df2.empty:
    print("Pipeline calisiyor!")
    display(df2[["departure_dt", "arrival_dt", "duration_min",
                 "price_eur", "stops"]].head(5))
else:
    print("UYARI: Sonuc yok — API erisimi veya tarih sorunu olabilir")

Arama: Frankfurt -> Berlin  |  Tarih: 2026-05-07

--- 1. CAGRI -----------------------------------------------
🚆 Train search: 'Frankfurt' → 'Berlin'  [DE]  date=2026-05-07
✅ DE train search: 5 results FRANKFURT(MAIN)→Berlin Hbf
💾 train cache SAVE frankfurt→berlin 2026-05-07 (5 rows)
  Sonuc: 5 sonuc

--- 2. CAGRI (onbellekten gelmeli) -------------------------
💾 train cache HIT  frankfurt→berlin 2026-05-07
  Sonuc: 5 sonuc

Pipeline calisiyor!


,departure_dt,arrival_dt,duration_min,price_eur,stops
0,2026-05-07 08:02:00,2026-05-07 12:01:00,239,99.99,0
1,2026-05-07 08:20:00,2026-05-07 12:38:00,258,99.99,0
2,2026-05-07 08:15:00,2026-05-07 12:30:00,255,119.99,0
3,2026-05-07 08:15:00,2026-05-07 12:22:00,247,133.99,1
4,2026-05-07 09:02:00,2026-05-07 12:54:00,232,147.99,0


In [12]:
# ─── Test 5: train_cache içeriğini incele ─────────────────────────────────
try:
    with sqlite3.connect(CACHE_DB) as conn:
        df_cache = pd.read_sql("""
            SELECT
                from_city,
                to_city,
                date,
                cached_at,
                length(payload)         AS payload_bytes,
                json_array_length(payload) AS row_count
            FROM train_cache
            ORDER BY cached_at DESC
        """, conn)

    if df_cache.empty:
        print("train_cache bos — once Test 3 veya Test 4 hucresini calistir")
    else:
        print(f"train_cache toplam: {len(df_cache)} kayit\n")
        display(df_cache)
except Exception as e:
    print(f"Cache DB okunamadi: {e}")

train_cache toplam: 7 kayit



,from_city,to_city,date,cached_at,payload_bytes,row_count
0,frankfurt,berlin,2026-05-07,2026-05-06T10:12:26.070076,3114,5
1,berlin,rome,2026-05-07,2026-05-06T10:10:56.428090,4051,7
2,milan,nurnberg,2026-05-05,2026-05-05T16:29:34.183672,2633,8
3,stuttgart,nurnberg,2026-05-05,2026-05-05T16:28:44.554368,1626,5
4,munich,nurnberg,2026-05-05,2026-05-05T16:28:44.442277,1721,5
5,venice,bologna,2026-05-05,2026-05-05T16:28:40.765199,3896,10
6,heringsdorf,firenze,2026-05-28,2026-05-05T16:18:38.810651,335,1


### Özet

| Test | Beklenen Sonuç |
|------|---------------|
| Test 1 — Station ID doğrulama | Tüm şehirler DB'de bulunmalı |
| Test 2 — Station ID ile arama | `_get_trips_de()` gerçek HAFAS sonuçları dönmeli |
| Test 3 — Cache yazma/okuma | `train_set` → `train_get` aynı DataFrame'i dönmeli |
| Test 4 — `get_trips()` pipeline | 2. çağrı "cache HIT" mesajı göstermeli |
| Test 5 — Cache içeriği | Yazılan kayıtlar tabloda görünmeli |

Tüm testler geçiyorsa tren arama + cache sistemi uygulama ile tam uyumludur.